In [1]:
import pandas as pd
import json
import glob
import os
import re

from functools import reduce

In [2]:
triples = pd.read_excel("../../outputs/clean_outputs/cv_triples_ISCO_ESCO_matches.xlsx")

In [3]:
triples.head()

,Unnamed: 0,candidate,triples,triples_top_matches_isco,triples_top_matches_esco
0,0,ebf358ed252344af8d1dd3f4c8516cbf,"[('ebf358ed252344af8d1dd3f4c8516cbf', 'has_wor...","[3222, 3421, 2166, 6129, 2529, 4221, 8132, 631...","['012924', '007756', '010923', '007348', '0072..."
1,1,6c6e7a728dcc4ade98e1367f4efe6fb7,"[('6c6e7a728dcc4ade98e1367f4efe6fb7', 'has_wor...","[8350, 3222, 9412, 3151, 2643, 5131, 3421, 441...","['008163', '010923', '005810', '004206', '0129..."
2,2,b1afb88436354245a405bcf36f4030c8,"[('b1afb88436354245a405bcf36f4030c8', 'has_wor...","[8132, 2166, 6310, 3222, 3421, 6129, 6112, 332...","['012924', '011116', '007756', '007348', '0081..."
3,3,e9e192123ed44166993497619c499bd5,"[('e9e192123ed44166993497619c499bd5', 'has_wor...","[3421, 4225, 3222, 4221, 8350, 5221, 5169, 711...","['005810', '006552', '002349', '010923', '0118..."
4,4,46353e9962684179846f77175feebef9,"[('46353e9962684179846f77175feebef9', 'has_wor...","[3222, 4415, 8350, 2529, 5142, 5131, 5169, 961...","['005810', '006335', '010818', '010026', '0111..."


In [4]:
# 1. Get a list of all your JSON files
file_pattern = './logs_isco_esco/temporary_results_cv_*.json' 
files = glob.glob(file_pattern)

# Step 1: Group all DataFrames by their model/prompt label
grouped_data = {}

for file in files:
    if os.path.getsize(file) == 0:
        continue

    try:
        with open(file, 'r') as f:
            data = json.load(f)
        
        temp_df = pd.DataFrame(data)
        
        # Extract label (e.g., 'qwen unstructured')
        basename = os.path.basename(file)
        name_match = re.search(r'results_(.*?)_\d', basename)
        label = name_match.group(1).rstrip('_').replace('_', ' ') if name_match else "unknown"

        if label not in grouped_data:
            grouped_data[label] = []
        
        grouped_data[label].append(temp_df)

    except Exception as e:
        print(f"Error reading {file}: {e}")

# Step 2: For each label, "squash" multiple files into one high-density DF
final_model_dfs = []

for label, dfs in grouped_data.items():
    # Stack all files for this model vertically
    combined = pd.concat(dfs, ignore_index=True)
    
    # Sort so that if there are duplicates, we have a consistent pick 
    # (Optional: sort by a timestamp if you want the newest values to take priority)
    # combined = combined.sort_values('some_timestamp_column', ascending=False)

    # The "Squash": Group by ID and take the first non-null value for ISCO and ESCO
    # This fills gaps where one file had IDs 0-500 and another had 500-1000
    squashed = combined.groupby('id', as_index=False).first()

    # Rename to your specific format
    rename_map = {
        'ISCO': f'ISCO {label}',
        'ESCO': f'ESCO {label}'
    }
    squashed = squashed.rename(columns=rename_map)
    
    # Keep only the ID and the new model-specific columns
    cols_to_keep = ['id', f'ISCO {label}', f'ESCO {label}']
    squashed = squashed[squashed.columns.intersection(cols_to_keep)]
    
    final_model_dfs.append(squashed)

# Step 3: Horizontal merge of the distinct models
if final_model_dfs:
    main_df = final_model_dfs[0]
    for next_df in final_model_dfs[1:]:
        main_df = pd.merge(main_df, next_df, on='id', how='outer')
    
    main_df = main_df.sort_values('id').reset_index(drop=True)
    
    print(f"Final Shape: {main_df.shape}")
    print("\nSample of merged columns:")
else:
    main_df = pd.DataFrame(columns=["candidate", "qwen", "gemma", "llama"])
    print("No data processed.")

No data processed.


In [5]:
main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   candidate  0 non-null      object
 1   qwen       0 non-null      object
 2   gemma      0 non-null      object
 3   llama      0 non-null      object
dtypes: object(4)
memory usage: 124.0+ bytes


In [6]:
df_int = pd.read_csv("../../../dataset/final_dataset/contacted_anon.csv")

# Only keep vacancies with at least 15 interactions
relevant_vacancies = df_int["cvid"].value_counts()[df_int["cvid"].value_counts() >= 15].index

# Filter to only relevant vacancies
df_int = df_int[df_int["cvid"].isin(relevant_vacancies.values)][["humanjobid", "cvid"]]

In [7]:
rel_vacancies = set(df_int["cvid"].values)
len(rel_vacancies)

1156

In [8]:
main_df["candidate"] = list(rel_vacancies)
main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1156 entries, 0 to 1155
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   candidate  1156 non-null   object
 1   qwen       0 non-null      object
 2   gemma      0 non-null      object
 3   llama      0 non-null      object
dtypes: object(4)
memory usage: 36.2+ KB


In [9]:
main_df.to_excel("../../outputs/raw_outputs/ISCO_ESCO_triples_cv.xlsx")

In [22]:
# 1. Identify all unique model/prompt combinations from the columns
# We look for columns starting with 'ISCO ' or 'ESCO '
columns = main_df.columns
model_prompt_pairs = set()

for col in columns:
    if col.startswith('ISCO '):
        model_prompt_pairs.add(col.replace('ISCO ', ''))

todo = {}

for pair in model_prompt_pairs:
    # Split "qwen unstructured" into model_name and prompt_type
    # We use rsplit to handle models that might have spaces in their name
    parts = pair.rsplit(' ', 1)
    model_name = parts[0]
    prompt_type = parts[1] if len(parts) > 1 else "default"
    
    # 2. Find IDs where ISCO or ESCO is null for this pair
    isco_col = f'ISCO {pair}'
    esco_col = f'ESCO {pair}'
    
    # Logic: An ID needs work if EITHER ISCO or ESCO is missing
    missing_mask = main_df[isco_col].isna() | main_df[esco_col].isna()
    missing_ids = main_df.loc[missing_mask, 'candidate'].tolist()
    
    # 3. Build the nested dictionary
    if model_name not in todo:
        todo[model_name] = {}
    
    todo[model_name][prompt_type] = missing_ids

# 4. Save to todo.json
with open('todo_ISCO_cv.json', 'w') as f:
    json.dump(todo, f)

print(f"todo.json created! Found {len(model_prompt_pairs)} model/prompt configurations.")

todo.json created! Found 9 model/prompt configurations.


In [11]:
# 1. Define the master lists based on your pipeline's needs
expected_models = ['qwen', 'gemma', 'llama'] 

# 2. Get the full list of all IDs (10,580 entries)
all_ids = main_df['candidate'].unique().tolist()

todo = {}

for model in expected_models:
    todo[model] = {}
    # Standardize the label used in columns: "model prompt"
    # We check both "model_prompt" and "model prompt" to be safe
    pair_label = f"{model}"
    isco_col = f'ISCO {pair_label}'
    esco_col = f'ESCO {pair_label}'
    
    # Check if we have any existing data for this combination
    if isco_col in main_df.columns:
        # Find IDs where either ISCO or ESCO is NaN
        missing_mask = main_df[isco_col].isna() | main_df[f'ESCO {pair_label}'].isna()
        missing_ids = main_df.loc[missing_mask, 'candidate'].tolist()
        todo[model] = missing_ids
    else:
        # If the model/prompt is totally missing, ALL IDs are todos
        todo[model] = all_ids

# 3. Save to todo.json
with open('todo_ISCO_cv.json', 'w') as f:
    json.dump(todo, f)

print(f"todo.json created. Included {len(expected_models)} models.")

todo.json created. Included 3 models.


In [13]:
for k, v in todo.items():
    print(k, len(v))

qwen 1156
gemma 1156
llama 1156
